In [1]:
import pandas as pd
import numpy as np

In [2]:
# Importamos el archivo json que contiene los datos de los usuarios de streaming
df = pd.read_json('../data/raw/streaming_users_dirty.json')

In [3]:
# Inspeccionamos el dataframe
df.info()
df.shape

<class 'pandas.DataFrame'>
RangeIndex: 8160 entries, 0 to 8159
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   user_id                   8160 non-null   int64  
 1   age                       8160 non-null   int64  
 2   subscription_plan         8160 non-null   str    
 3   monthly_watch_time_mins   7967 non-null   float64
 4   country                   8160 non-null   str    
 5   favorite_genre            7920 non-null   str    
 6   last_login_date           7840 non-null   str    
 7   customer_support_tickets  8160 non-null   int64  
dtypes: float64(1), int64(3), str(4)
memory usage: 756.7 KB


(8160, 8)

El dataset cuenta con 8 columnas y 8160 filas, se detecta que fecha esta como str y no como date

In [4]:
df.describe()

,user_id,age,monthly_watch_time_mins,customer_support_tickets
count,8160.000000,8160.000000,7967.000000,8160.000000
mean,13995.433824,34.096814,1107.346153,1.800980
std,2310.810660,14.511304,5310.442604,11.334969
min,10000.000000,-5.000000,-120.000000,-1.000000
25%,11987.750000,25.000000,489.200000,0.000000
50%,13998.500000,33.000000,757.400000,1.000000
75%,15997.250000,42.000000,1045.700000,1.000000
max,17999.000000,150.000000,99999.000000,150.000000


Aqui detectamos: min y max en 'age', 'monthly_watch_time_mins' y 'customer_support_tickets' que deben ser irreales.

In [5]:
# Detectamos los valores nulos en el dataframe
df.isnull().sum()

user_id                       0
age                           0
subscription_plan             0
monthly_watch_time_mins     193
country                       0
favorite_genre              240
last_login_date             320
customer_support_tickets      0
dtype: int64

In [6]:
df.head(10)

,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
0,10000,39,Estándar,805.8,Brasil,Crime,2025-03-04,99
1,10001,37,Estándar,1173.4,Colombia,Crime,2019-04-02,2
2,10002,28,Básico,401.0,Colombia,Crime,2018-04-13,0
3,10003,43,Básico,62.4,Uruguay,Thriller,2021-01-31,0
4,10004,51,Básico,477.8,Perú,Thriller,2020-09-30,1
5,10005,20,Básico,670.2,Uruguay,Drama,2020-07-03,2
6,10006,37,Básico,346.6,Perú,Thriller,2019-07-26,1
7,10007,31,Estándar,974.6,Chile,Acción,2019-02-24,1
8,10008,36,Premium,1432.2,Colombia,Romance,2025-08-03,2
9,10009,37,Estándar,1375.4,Argentina,Thriller,2024-02-12,1


In [7]:
df.duplicated().sum()

np.int64(126)

Se detectaron 126 registros duplicados absolutos que deberán ser tratados en la etapa de limpieza

In [8]:
# Detectamos los duplicados en el dataframe
df[df.duplicated(keep=False)].sort_values(by=df.columns.tolist())

,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
37,10037,33,Básico,611.0,Colombia,Documental,2019-09-29,2
8133,10037,33,Básico,611.0,Colombia,Documental,2019-09-29,2
52,10052,25,Básico,465.7,Colombia,Acción,2022-03-31,1
8089,10052,25,Básico,465.7,Colombia,Acción,2022-03-31,1
117,10117,29,Estándar,713.9,Brasil,Documental,2020-12-20,1
...,...,...,...,...,...,...,...,...
8115,17684,38,Premium,1283.2,México,Comedia,2024-09-13,0
7784,17784,38,Estándar,492.2,México,Documental,2021-10-04,1
8125,17784,38,Estándar,492.2,México,Documental,2021-10-04,1
7994,17994,24,Básico,437.0,México,Romance,2020-08-12,0


In [9]:
# Filtramos todos los registros donde la columna específica se repite
df.duplicated(subset=['user_id']).sum()

np.int64(160)

In [10]:
columna_objetivo = 'user_id'

# 1. Filtramos todos los registros donde la columna específica se repite
duplicados_por_columna = df[df.duplicated(subset=[columna_objetivo], keep=False)]

# 2. Excluimos los clones absolutos (la tilde ~ invierte la condición booleana)
duplicados_parciales = duplicados_por_columna[~duplicados_por_columna.duplicated(keep=False)]

# 3. Mostramos la totalidad de las filas conflictivas ordenadas para su comparación
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(duplicados_parciales.sort_values(by=[columna_objetivo]))

,user_id,age,subscription_plan,monthly_watch_time_mins,country,favorite_genre,last_login_date,customer_support_tickets
59,10059,39,Estándar,2976.6,Brasil,DRAMA,2024-06-22,1
8090,10059,39,Estándar,824.8,Brasil,Drama,2024-06-22,1
8126,10721,32,Estándar,959.0,Colombia,Documental,2021-09-02,2
721,10721,32,Estándar,959.0,Colombia,Documental,NaN,2
797,10797,31,Básico,-1.0,México,Comedia,2023-07-01,1
8018,10797,31,Básico,410.4,México,Comedia,2023-07-01,1
8076,11092,31,Estándar,959.6,Chile,Romance,2025-01-05,2
1092,11092,31,Estándar,959.6,Chile,Romance,2025-01-05,2
1222,11222,13,Estándar,1321.8,MEX,Documental,2019-02-08,0
8157,11222,13,Estándar,1321.8,México,Documental,2019-02-08,0


In [11]:
# Detectamos los valores únicos de la columna 'country'
df['country'].unique()

<ArrowStringArray>
[    'Brasil',   'Colombia',    'Uruguay',       'Perú',      'Chile',
  'Argentina',     'México',     'Brazil',     'brasil',     'méxico',
      'chile',    'uruguay',        'MEX',        'ARG',   'colombia',
        'COL',     'Mexico',        'URY',     'Chile ',       'Peru',
       'perú',  'argentina',        'PER', 'Argentina ',        'BRA',
        'CHL']
Length: 26, dtype: str

In [12]:
# Detectamos los valores únicos de la columna 'favorite_genre'
df['favorite_genre'].unique()

<ArrowStringArray>
[      'Crime',    'Thriller',       'Drama',      'Acción',     'Romance',
     'Comedia',  'Documental',      'ACCIÓN',           nan,       'CRIME',
    'Comedia ',      'comedy',       'DRAMA',    'THRILLER', 'Documentary',
   'Thriller ',      'accion',      'Action',     'COMEDIA',     'ROMANCE',
     'thriler',         'DOC',     'romance',      'Crimen',      'Drama ',
       'drama',    'Romance ',  'documental',       'crime']
Length: 29, dtype: str

In [13]:
df['subscription_plan'].unique()

<ArrowStringArray>
[ 'Estándar',    'Básico',   'Premium',       'Std',  'estandar',    'basico',
    'básico',  'Premium ',   'premium',   'Premiun',    'BASICO',  'STANDARD',
     'Basic', 'Estándar ',   'PREMIUM']
Length: 15, dtype: str

En el analisis detecte que hay duplicados exactos en todas las filas (126) pero el ID se repite 160 veces, por lo que hay 34 registros que son coincidencias parciales. Ademas vemos las variantes de cada fila tipo string para noralizar.